<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_1_state_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/llm"

RAW_DIR = f"{PROJECT_DIR}/data_raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"
STATE_DIR = f"{PROJECT_DIR}/patient_states"
REPORT_DIR = f"{PROJECT_DIR}/reports"
UTILS_DIR = f"{PROJECT_DIR}/utils"

In [ ]:
import os

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    STATE_DIR,
    REPORT_DIR,
    UTILS_DIR
]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
!unzip '/content/drive/MyDrive/Colab Datasets/hf.zip' -d {RAW_DIR}

In [ ]:
import pandas as pd

dat = pd.read_csv(f"{RAW_DIR}/dat.csv")
dat_md = pd.read_csv(f"{RAW_DIR}/dat_md.csv")
data_dict = pd.read_csv(f"{RAW_DIR}/dataDictionary.csv")

In [ ]:
print("dat.csv shape:", dat.shape)
print("dat_md.csv shape:", dat_md.shape)
print("dataDictionary.csv shape:", data_dict.shape)


dat.csv shape: (2008, 167)
dat_md.csv shape: (15362, 3)
dataDictionary.csv shape: (166, 3)


In [ ]:

display(dat.head())
display(dat_md.head())
display(data_dict.head())

In [ ]:
def drop_unnamed_columns(df):
    unnamed_cols = [c for c in df.columns if c.startswith("Unnamed")]
    return df.drop(columns=unnamed_cols), unnamed_cols

dat, dat_dropped_cols = drop_unnamed_columns(dat)
dat_md, med_dropped_cols = drop_unnamed_columns(dat_md)
data_dict, dict_dropped_cols = drop_unnamed_columns(data_dict)

print("Dropped from dat:", dat_dropped_cols)
print("Dropped from dat_md:", med_dropped_cols)
print("Dropped from dataDictionary:", dict_dropped_cols)

print("Cleaned dat shape:", dat.shape)
print("Cleaned dat_md shape:", dat_md.shape)
print("Cleaned data_dict shape:", data_dict.shape)

In [ ]:
import json

In [ ]:
dict_vars = set(data_dict["Variables"].astype(str))
dat_vars = set(dat.columns)

missing_in_dictionary = sorted(list(dat_vars - dict_vars))
missing_in_dat = sorted(list(dict_vars - dat_vars))

schema_check = {
    "dat_columns": len(dat.columns),
    "dictionary_variables": len(data_dict),
    "columns_missing_in_dictionary": missing_in_dictionary,
    "dictionary_variables_missing_in_dat": missing_in_dat
}

schema_check_path = f"{REPORT_DIR}/schema_check.json"

with open(schema_check_path, "w") as f:
    json.dump(schema_check, f, indent=2)

schema_check

{'dat_columns': 166,
 'dictionary_variables': 166,
 'columns_missing_in_dictionary': [],
 'dictionary_variables_missing_in_dat': []}

In [ ]:
description_map = dict(
    zip(data_dict["Variables"].astype(str), data_dict["Description"].astype(str))
)

def column_profile(df):
    rows = []

    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)
        missing_count = int(s.isna().sum())
        missing_pct = round(float(s.isna().mean() * 100), 3)
        unique_count = int(s.nunique(dropna=True))

        examples = (
            s.dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )

        row = {
            "column": col,
            "description": description_map.get(col, ""),
            "dtype": dtype,
            "missing_count": missing_count,
            "missing_pct": missing_pct,
            "unique_count": unique_count,
            "examples": " | ".join(examples)
        }

        if pd.api.types.is_numeric_dtype(s):
            row.update({
                "min": s.min(skipna=True),
                "p25": s.quantile(0.25),
                "median": s.quantile(0.50),
                "p75": s.quantile(0.75),
                "max": s.max(skipna=True),
                "zero_count": int((s == 0).sum())
            })
        else:
            row.update({
                "min": None,
                "p25": None,
                "median": None,
                "p75": None,
                "max": None,
                "zero_count": None
            })

        rows.append(row)

    return pd.DataFrame(rows)

column_summary = column_profile(dat)

column_summary_path = f"{REPORT_DIR}/column_summary.csv"
column_summary.to_csv(column_summary_path, index=False)

display(column_summary)
print("Saved:", column_summary_path)

In [ ]:
missingness_report = column_summary[
    ["column", "description", "dtype", "missing_count", "missing_pct", "unique_count"]
].sort_values("missing_pct", ascending=False)

missingness_path = f"{REPORT_DIR}/missingness_report.csv"
missingness_report.to_csv(missingness_path, index=False)

display(missingness_report.head(40))
print("Saved:", missingness_path)

In [ ]:
categorical_cols = dat.select_dtypes(include=["object"]).columns.tolist()

cat_rows = []

for col in categorical_cols:
    vc = dat[col].value_counts(dropna=False)

    cat_rows.append({
        "column": col,
        "description": description_map.get(col, ""),
        "missing_count": int(dat[col].isna().sum()),
        "missing_pct": round(float(dat[col].isna().mean() * 100), 3),
        "unique_count": int(dat[col].nunique(dropna=True)),
        "top_values": " | ".join([f"{idx}: {val}" for idx, val in vc.head(10).items()])
    })

categorical_report = pd.DataFrame(cat_rows)

categorical_path = f"{REPORT_DIR}/categorical_report.csv"
categorical_report.to_csv(categorical_path, index=False)

display(categorical_report)
print("Saved:", categorical_path)

In [ ]:
outcome_cols = [
    "outcome.during.hospitalization",
    "death.within.28.days",
    "re.admission.within.28.days",
    "death.within.3.months",
    "re.admission.within.3.months",
    "death.within.6.months",
    "re.admission.within.6.months",
    "return.to.emergency.department.within.6.months",
    "time.of.death..days.from.admission.",
    "re.admission.time..days.from.admission.",
    "time.to.emergency.department.within.6.months"
]

outcome_rows = []

for col in outcome_cols:
    if col in dat.columns:
        vc = dat[col].value_counts(dropna=False)

        outcome_rows.append({
            "column": col,
            "description": description_map.get(col, ""),
            "missing_count": int(dat[col].isna().sum()),
            "missing_pct": round(float(dat[col].isna().mean() * 100), 3),
            "value_distribution": " | ".join([f"{idx}: {val}" for idx, val in vc.items()])
        })

outcome_summary = pd.DataFrame(outcome_rows)

outcome_path = f"{REPORT_DIR}/outcome_summary.csv"
outcome_summary.to_csv(outcome_path, index=False)

display(outcome_summary)
print("Saved:", outcome_path)

In [ ]:
PRIMARY_TARGET = "re.admission.within.28.days"

In [ ]:
print("Medication table shape:", dat_md.shape)
print("Medication columns:", dat_md.columns.tolist())

patient_ids = set(dat["inpatient.number"])
med_patient_ids = set(dat_md["inpatient.number"])

linkage_report = {
    "patients_in_dat": len(patient_ids),
    "patients_in_med_table": len(med_patient_ids),
    "patients_with_medication_records": len(patient_ids & med_patient_ids),
    "patients_without_medication_records": len(patient_ids - med_patient_ids),
    "medication_patient_ids_not_in_dat": len(med_patient_ids - patient_ids),
    "total_medication_rows": len(dat_md),
    "unique_drugs": int(dat_md["Drug_name"].nunique())
}

with open(f"{REPORT_DIR}/medication_linkage_report.json", "w") as f:
    json.dump(linkage_report, f, indent=2)

linkage_report

Medication table shape: (15362, 2)
Medication columns: ['inpatient.number', 'Drug_name']


{'patients_in_dat': 2008,
 'patients_in_med_table': 2007,
 'patients_with_medication_records': 2007,
 'patients_without_medication_records': 1,
 'medication_patient_ids_not_in_dat': 0,
 'total_medication_rows': 15362,
 'unique_drugs': 25}

In [ ]:
drug_frequency = (
    dat_md["Drug_name"]
    .value_counts()
    .reset_index()
)

drug_frequency.columns = ["Drug_name", "count"]

drug_frequency_path = f"{REPORT_DIR}/drug_frequency.csv"
drug_frequency.to_csv(drug_frequency_path, index=False)

med_count_per_patient = (
    dat_md
    .groupby("inpatient.number")
    .size()
    .reset_index(name="med_count")
)

med_count_summary = med_count_per_patient["med_count"].describe().reset_index()
med_count_summary.columns = ["statistic", "value"]

med_count_summary_path = f"{REPORT_DIR}/medication_count_summary.csv"
med_count_summary.to_csv(med_count_summary_path, index=False)

display(drug_frequency)
display(med_count_summary)

print("Saved:", drug_frequency_path)
print("Saved:", med_count_summary_path)

In [ ]:
FEATURE_GROUPS = {
    "id": [
        "inpatient.number"
    ],

    "demographics": [
        "gender",
        "ageCat",
        "occupation"
    ],

    "encounter_context": [
        "admission.way",
        "admission.ward",
        "visit.times",
        "discharge.department",
        "dischargeDay"
    ],

    "vitals_body": [
        "body.temperature",
        "pulse",
        "respiration",
        "systolic.blood.pressure",
        "diastolic.blood.pressure",
        "map",
        "weight",
        "height",
        "BMI"
    ],

    "heart_failure_severity": [
        "type.of.heart.failure",
        "NYHA.cardiac.function.classification",
        "Killip.grade",
        "LVEF",
        "left.ventricular.end.diastolic.diameter.LV"
    ],

    "comorbidities": [
        "myocardial.infarction",
        "congestive.heart.failure",
        "peripheral.vascular.disease",
        "cerebrovascular.disease",
        "dementia",
        "Chronic.obstructive.pulmonary.disease",
        "connective.tissue.disease",
        "peptic.ulcer.disease",
        "diabetes",
        "moderate.to.severe.chronic.kidney.disease",
        "hemiplegia",
        "leukemia",
        "malignant.lymphoma",
        "solid.tumor",
        "liver.disease",
        "AIDS",
        "CCI.score"
    ],

    "respiratory_neuro_status": [
        "type.II.respiratory.failure",
        "consciousness",
        "eye.opening",
        "verbal.response",
        "movement",
        "GCS",
        "oxygen.inhalation",
        "fio2",
        "acute.renal.failure"
    ],

    "renal_labs": [
        "creatinine.enzymatic.method",
        "urea",
        "uric.acid",
        "glomerular.filtration.rate",
        "cystatin"
    ],

    "cbc_labs": [
        "white.blood.cell",
        "red.blood.cell",
        "hemoglobin",
        "platelet",
        "hematocrit",
        "neutrophil.ratio",
        "lymphocyte.count"
    ],

    "electrolytes": [
        "sodium",
        "potassium",
        "chloride",
        "calcium"
    ],

    "cardiac_biomarkers": [
        "brain.natriuretic.peptide",
        "high.sensitivity.troponin",
        "creatine.kinase",
        "creatine.kinase.isoenzyme",
        "lactate.dehydrogenase"
    ],

    "coagulation": [
        "D.dimer",
        "international.normalized.ratio",
        "activated.partial.thromboplastin.time",
        "prothrombin.time.ratio",
        "fibrinogen"
    ],

    "nutrition_liver": [
        "albumin",
        "globulin",
        "total.protein",
        "direct.bilirubin",
        "total.bilirubin",
        "alkaline.phosphatase",
        "glutamyltranspeptidase",
        "glutamic.pyruvic.transaminase",
        "glutamic.oxaloacetic.transaminase"
    ],

    "labels": [
        "death.within.28.days",
        "re.admission.within.28.days",
        "death.within.3.months",
        "re.admission.within.3.months",
        "death.within.6.months",
        "re.admission.within.6.months",
        "return.to.emergency.department.within.6.months"
    ]
}

selected_features = []

for group, cols in FEATURE_GROUPS.items():
    for col in cols:
        if col in dat.columns and col not in selected_features:
            selected_features.append(col)

selected_features_path = f"{REPORT_DIR}/selected_features.json"

with open(selected_features_path, "w") as f:
    json.dump({
        "feature_groups": FEATURE_GROUPS,
        "selected_features": selected_features
    }, f, indent=2)

print("Number of selected columns:", len(selected_features))
print("Saved:", selected_features_path)

Number of selected columns: 91
Saved: /content/drive/MyDrive/llm/reports/selected_features.json


In [ ]:


import os
import json
import pandas as pd
import numpy as np
import re

PROJECT_DIR = "/content/drive/MyDrive/llm"

RAW_DIR = f"{PROJECT_DIR}/data_raw"
REPORT_DIR = f"{PROJECT_DIR}/reports"
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

DAT_PATH = f"{RAW_DIR}/dat.csv"
MED_PATH = f"{RAW_DIR}/dat_md.csv"
SELECTED_FEATURES_PATH = f"{REPORT_DIR}/selected_features.json"

In [ ]:
dat = pd.read_csv(DAT_PATH)
dat_md = pd.read_csv(MED_PATH)

dat = dat.drop(columns=[c for c in dat.columns if c.startswith("Unnamed")])
dat_md = dat_md.drop(columns=[c for c in dat_md.columns if c.startswith("Unnamed")])

with open(SELECTED_FEATURES_PATH, "r") as f:
    feature_info = json.load(f)

FEATURE_GROUPS = feature_info["feature_groups"]
selected_features = feature_info["selected_features"]

selected_features = [c for c in selected_features if c in dat.columns]

patient_df = dat[selected_features].copy()

print("Selected patient table shape:", patient_df.shape)
display(patient_df.head())

In [ ]:
RANGE_RULES = {
    "body.temperature": (30, 45),
    "pulse": (20, 250),
    "respiration": (5, 60),
    "systolic.blood.pressure": (50, 300),
    "diastolic.blood.pressure": (30, 180),
    "map": (40, 220),
    "weight": (20, 250),
    "height": (1.0, 2.2),
    "BMI": (10, 80),
    "GCS": (3, 15),
    "LVEF": (5, 90),
    "left.ventricular.end.diastolic.diameter.LV": (20, 100),
    "sodium": (90, 180),
    "potassium": (1.0, 12.0),
    "chloride": (60, 140),
    "calcium": (1.0, 4.0),
    "creatinine.enzymatic.method": (10, 1500),
    "urea": (0.5, 80),
    "glomerular.filtration.rate": (1, 300),
    "brain.natriuretic.peptide": (0, 100000),
    "high.sensitivity.troponin": (0, 100),
    "white.blood.cell": (0.1, 100),
    "hemoglobin": (20, 250),
    "platelet": (1, 1000),
    "albumin": (5, 80),
    "dischargeDay": (1, 365)
}

cleaning_log = []

for col, (low, high) in RANGE_RULES.items():
    if col in patient_df.columns:
        before_missing = patient_df[col].isna().sum()

        invalid_mask = (
            patient_df[col].notna()
            & ((patient_df[col] < low) | (patient_df[col] > high))
        )

        invalid_count = int(invalid_mask.sum())
        patient_df.loc[invalid_mask, col] = np.nan

        after_missing = patient_df[col].isna().sum()

        cleaning_log.append({
            "column": col,
            "allowed_min": low,
            "allowed_max": high,
            "invalid_values_set_to_missing": invalid_count,
            "missing_before": int(before_missing),
            "missing_after": int(after_missing)
        })

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(f"{PROCESSED_DIR}/cleaning_log.csv", index=False)

display(cleaning_log_df)

In [ ]:
categorical_cols = patient_df.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    patient_df[col] = patient_df[col].fillna("Unknown")

display(patient_df[categorical_cols].head())

In [ ]:
def classify_medication(drug_name):
    name = str(drug_name).lower()

    classes = []

    if any(x in name for x in ["furosemide", "torasemide", "hydrochlorothiazide", "spironolactone"]):
        classes.append("diuretic_or_mra")

    if any(x in name for x in ["benazepril", "valsartan"]):
        classes.append("acei_or_arb")

    if "metoprolol" in name:
        classes.append("beta_blocker")

    if any(x in name for x in ["digoxin", "deslanoside"]):
        classes.append("cardiac_glycoside")

    if any(x in name for x in ["aspirin", "clopidogrel"]):
        classes.append("antiplatelet")

    if any(x in name for x in ["warfarin", "heparin", "enoxaparin"]):
        classes.append("anticoagulant")

    if any(x in name for x in ["nitroglycerin", "isosorbide"]):
        classes.append("nitrate")

    if any(x in name for x in ["milrinone", "dobutamine", "isoprenaline"]):
        classes.append("inotrope_or_pressor")

    if "atorvastatin" in name:
        classes.append("statin")

    if len(classes) == 0:
        classes.append("other")

    return classes

med_temp = dat_md.copy()
med_temp["Drug_name"] = med_temp["Drug_name"].astype(str)
med_temp["med_classes"] = med_temp["Drug_name"].apply(classify_medication)

med_summary = (
    med_temp
    .groupby("inpatient.number")
    .agg(
        med_count=("Drug_name", "count"),
        unique_med_count=("Drug_name", "nunique"),
        med_list=("Drug_name", lambda x: " | ".join(sorted(set(x)))),
    )
    .reset_index()
)

all_classes = sorted(set(cls for classes in med_temp["med_classes"] for cls in classes))

for med_class in all_classes:
    ids_with_class = med_temp[
        med_temp["med_classes"].apply(lambda classes: med_class in classes)
    ]["inpatient.number"].unique()

    med_summary[f"has_{med_class}"] = med_summary["inpatient.number"].isin(ids_with_class).astype(int)

med_summary_path = f"{PROCESSED_DIR}/patient_med_summary.csv"
med_summary.to_csv(med_summary_path, index=False)

display(med_summary.head())
print("Saved:", med_summary_path)

In [ ]:
patient_master = patient_df.merge(
    med_summary,
    on="inpatient.number",
    how="left"
)

patient_master["med_count"] = patient_master["med_count"].fillna(0).astype(int)
patient_master["unique_med_count"] = patient_master["unique_med_count"].fillna(0).astype(int)
patient_master["med_list"] = patient_master["med_list"].fillna("")

for col in patient_master.columns:
    if col.startswith("has_"):
        patient_master[col] = patient_master[col].fillna(0).astype(int)

print("Patient master shape:", patient_master.shape)
display(patient_master.head())

In [ ]:
label_cols = [
    "inpatient.number",
    "death.within.28.days",
    "re.admission.within.28.days",
    "death.within.3.months",
    "re.admission.within.3.months",
    "death.within.6.months",
    "re.admission.within.6.months",
    "return.to.emergency.department.within.6.months"
]

label_cols = [c for c in label_cols if c in patient_master.columns]

patient_labels = patient_master[label_cols].copy()

patient_labels = patient_labels.rename(columns={
    "death.within.28.days": "death_28d",
    "re.admission.within.28.days": "readmission_28d",
    "death.within.3.months": "death_3m",
    "re.admission.within.3.months": "readmission_3m",
    "death.within.6.months": "death_6m",
    "re.admission.within.6.months": "readmission_6m",
    "return.to.emergency.department.within.6.months": "ed_return_6m"
})

display(patient_labels.head())

patient_labels["readmission_28d"].value_counts(dropna=False)

In [ ]:
patient_master_path = f"{PROCESSED_DIR}/patient_master_table.csv"
patient_labels_path = f"{PROCESSED_DIR}/patient_labels.csv"

patient_master.to_csv(patient_master_path, index=False)
patient_labels.to_csv(patient_labels_path, index=False)

print("Saved:", patient_master_path)
print("Saved:", patient_labels_path)
print("Saved:", med_summary_path)

Saved: /content/drive/MyDrive/llm/data_processed/patient_master_table.csv
Saved: /content/drive/MyDrive/llm/data_processed/patient_labels.csv
Saved: /content/drive/MyDrive/llm/data_processed/patient_med_summary.csv


In [ ]:
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"
STATE_DIR = f"{PROJECT_DIR}/patient_states"

os.makedirs(STATE_DIR, exist_ok=True)

PATIENT_MASTER_PATH = f"{PROCESSED_DIR}/patient_master_table.csv"
PATIENT_LABELS_PATH = f"{PROCESSED_DIR}/patient_labels.csv"

In [ ]:
patient_master = pd.read_csv(PATIENT_MASTER_PATH)
patient_labels = pd.read_csv(PATIENT_LABELS_PATH)

print("Patient master shape:", patient_master.shape)
print("Patient labels shape:", patient_labels.shape)

display(patient_master.head())
display(patient_labels.head())

In [ ]:
def clean_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value


def get_value(row, col):
    if col not in row.index:
        return None

    return clean_value(row[col])


def split_med_list(med_string):
    if pd.isna(med_string) or str(med_string).strip() == "":
        return []

    return [m.strip() for m in str(med_string).split("|") if m.strip()]


COMORBIDITY_COLUMNS = {
    "myocardial.infarction": "myocardial infarction",
    "congestive.heart.failure": "congestive heart failure",
    "peripheral.vascular.disease": "peripheral vascular disease",
    "cerebrovascular.disease": "cerebrovascular disease",
    "dementia": "dementia",
    "Chronic.obstructive.pulmonary.disease": "chronic obstructive pulmonary disease",
    "connective.tissue.disease": "connective tissue disease",
    "peptic.ulcer.disease": "peptic ulcer disease",
    "diabetes": "diabetes",
    "moderate.to.severe.chronic.kidney.disease": "moderate to severe chronic kidney disease",
    "hemiplegia": "hemiplegia",
    "leukemia": "leukemia",
    "malignant.lymphoma": "malignant lymphoma",
    "solid.tumor": "solid tumor",
    "liver.disease": "liver disease",
    "AIDS": "AIDS"
}


def extract_comorbidities(row):
    present = []

    for col, label in COMORBIDITY_COLUMNS.items():
        value = get_value(row, col)

        if value == 1:
            present.append(label)

    return present

In [ ]:
def build_patient_state(row):
    medications = split_med_list(get_value(row, "med_list"))

    state = {
        "patient_id": get_value(row, "inpatient.number"),

        "patient_context": {
            "demographics": {
                "gender": get_value(row, "gender"),
                "age_category": get_value(row, "ageCat"),
                "occupation": get_value(row, "occupation")
            },

            "encounter_context": {
                "admission_way": get_value(row, "admission.way"),
                "admission_ward": get_value(row, "admission.ward"),
                "visit_times": get_value(row, "visit.times"),
                "discharge_department": get_value(row, "discharge.department"),
                "length_of_stay_days": get_value(row, "dischargeDay")
            },

            "heart_failure_status": {
                "type_of_heart_failure": get_value(row, "type.of.heart.failure"),
                "nyha_class": get_value(row, "NYHA.cardiac.function.classification"),
                "killip_grade": get_value(row, "Killip.grade"),
                "lvef": get_value(row, "LVEF"),
                "lv_end_diastolic_diameter": get_value(row, "left.ventricular.end.diastolic.diameter.LV")
            },

            "vitals_and_body": {
                "body_temperature_c": get_value(row, "body.temperature"),
                "pulse_bpm": get_value(row, "pulse"),
                "respiration_rate": get_value(row, "respiration"),
                "systolic_bp": get_value(row, "systolic.blood.pressure"),
                "diastolic_bp": get_value(row, "diastolic.blood.pressure"),
                "mean_arterial_pressure": get_value(row, "map"),
                "weight_kg": get_value(row, "weight"),
                "height_m": get_value(row, "height"),
                "bmi": get_value(row, "BMI")
            },

            "comorbidities": {
                "cci_score": get_value(row, "CCI.score"),
                "present_comorbidities": extract_comorbidities(row)
            },

            "respiratory_neurologic_status": {
                "type_ii_respiratory_failure": get_value(row, "type.II.respiratory.failure"),
                "consciousness": get_value(row, "consciousness"),
                "eye_opening": get_value(row, "eye.opening"),
                "verbal_response": get_value(row, "verbal.response"),
                "movement": get_value(row, "movement"),
                "gcs": get_value(row, "GCS"),
                "oxygen_inhalation": get_value(row, "oxygen.inhalation"),
                "fio2": get_value(row, "fio2"),
                "acute_renal_failure": get_value(row, "acute.renal.failure")
            },

            "key_labs": {
                "renal": {
                    "creatinine_enzymatic_method": get_value(row, "creatinine.enzymatic.method"),
                    "urea": get_value(row, "urea"),
                    "uric_acid": get_value(row, "uric.acid"),
                    "glomerular_filtration_rate": get_value(row, "glomerular.filtration.rate"),
                    "cystatin": get_value(row, "cystatin")
                },

                "electrolytes": {
                    "sodium": get_value(row, "sodium"),
                    "potassium": get_value(row, "potassium"),
                    "chloride": get_value(row, "chloride"),
                    "calcium": get_value(row, "calcium")
                },

                "cbc": {
                    "white_blood_cell": get_value(row, "white.blood.cell"),
                    "red_blood_cell": get_value(row, "red.blood.cell"),
                    "hemoglobin": get_value(row, "hemoglobin"),
                    "platelet": get_value(row, "platelet"),
                    "hematocrit": get_value(row, "hematocrit"),
                    "neutrophil_ratio": get_value(row, "neutrophil.ratio"),
                    "lymphocyte_count": get_value(row, "lymphocyte.count")
                },

                "cardiac_biomarkers": {
                    "brain_natriuretic_peptide": get_value(row, "brain.natriuretic.peptide"),
                    "high_sensitivity_troponin": get_value(row, "high.sensitivity.troponin"),
                    "creatine_kinase": get_value(row, "creatine.kinase"),
                    "creatine_kinase_isoenzyme": get_value(row, "creatine.kinase.isoenzyme"),
                    "lactate_dehydrogenase": get_value(row, "lactate.dehydrogenase")
                },

                "coagulation": {
                    "d_dimer": get_value(row, "D.dimer"),
                    "international_normalized_ratio": get_value(row, "international.normalized.ratio"),
                    "activated_partial_thromboplastin_time": get_value(row, "activated.partial.thromboplastin.time"),
                    "prothrombin_time_ratio": get_value(row, "prothrombin.time.ratio"),
                    "fibrinogen": get_value(row, "fibrinogen")
                },

                "nutrition_liver": {
                    "albumin": get_value(row, "albumin"),
                    "globulin": get_value(row, "globulin"),
                    "total_protein": get_value(row, "total.protein"),
                    "direct_bilirubin": get_value(row, "direct.bilirubin"),
                    "total_bilirubin": get_value(row, "total.bilirubin"),
                    "alkaline_phosphatase": get_value(row, "alkaline.phosphatase")
                }
            },

            "medications": {
                "med_count": get_value(row, "med_count"),
                "unique_med_count": get_value(row, "unique_med_count"),
                "medication_names": medications,
                "has_diuretic_or_mra": get_value(row, "has_diuretic_or_mra"),
                "has_acei_or_arb": get_value(row, "has_acei_or_arb"),
                "has_beta_blocker": get_value(row, "has_beta_blocker"),
                "has_cardiac_glycoside": get_value(row, "has_cardiac_glycoside"),
                "has_antiplatelet": get_value(row, "has_antiplatelet"),
                "has_anticoagulant": get_value(row, "has_anticoagulant"),
                "has_nitrate": get_value(row, "has_nitrate"),
                "has_inotrope_or_pressor": get_value(row, "has_inotrope_or_pressor"),
                "has_statin": get_value(row, "has_statin")
            }
        },

        "evaluation_labels": {
            "death_28d": get_value(row, "death.within.28.days"),
            "readmission_28d": get_value(row, "re.admission.within.28.days"),
            "death_3m": get_value(row, "death.within.3.months"),
            "readmission_3m": get_value(row, "re.admission.within.3.months"),
            "death_6m": get_value(row, "death.within.6.months"),
            "readmission_6m": get_value(row, "re.admission.within.6.months"),
            "ed_return_6m": get_value(row, "return.to.emergency.department.within.6.months")
        }
    }

    return state


patient_states = []

for _, row in patient_master.iterrows():
    patient_states.append(build_patient_state(row))

print("Number of patient states:", len(patient_states))
patient_states[0]

In [ ]:
patient_states_path = f"{STATE_DIR}/patient_states.json"

with open(patient_states_path, "w") as f:
    json.dump(patient_states, f, indent=2)

print("Saved:", patient_states_path)

Saved: /content/drive/MyDrive/llm/patient_states/patient_states.json


In [ ]:
def fmt(value):
    if value is None or pd.isna(value):
        return "not available"

    return str(value)


def patient_state_to_text(state):
    ctx = state["patient_context"]

    demographics = ctx["demographics"]
    encounter = ctx["encounter_context"]
    hf = ctx["heart_failure_status"]
    vitals = ctx["vitals_and_body"]
    comorb = ctx["comorbidities"]
    resp = ctx["respiratory_neurologic_status"]
    labs = ctx["key_labs"]
    meds = ctx["medications"]

    med_names = meds.get("medication_names", [])
    med_text = ", ".join(med_names) if len(med_names) > 0 else "not available"

    comorb_list = comorb.get("present_comorbidities", [])
    comorb_text = ", ".join(comorb_list) if len(comorb_list) > 0 else "none recorded"

    text = f"""
Patient State Summary:
- Patient ID: {fmt(state["patient_id"])}
- Gender: {fmt(demographics["gender"])}
- Age category: {fmt(demographics["age_category"])}
- Occupation: {fmt(demographics["occupation"])}

Encounter / discharge context:
- Admission way: {fmt(encounter["admission_way"])}
- Admission ward: {fmt(encounter["admission_ward"])}
- Previous visit count: {fmt(encounter["visit_times"])}
- Discharge department: {fmt(encounter["discharge_department"])}
- Length of stay: {fmt(encounter["length_of_stay_days"])} days

Heart failure status:
- Type of heart failure: {fmt(hf["type_of_heart_failure"])}
- NYHA class: {fmt(hf["nyha_class"])}
- Killip grade: {fmt(hf["killip_grade"])}
- LVEF: {fmt(hf["lvef"])}
- LV end-diastolic diameter: {fmt(hf["lv_end_diastolic_diameter"])}

Vitals/body:
- Temperature: {fmt(vitals["body_temperature_c"])} °C
- Pulse: {fmt(vitals["pulse_bpm"])} bpm
- Respiratory rate: {fmt(vitals["respiration_rate"])}
- Blood pressure: {fmt(vitals["systolic_bp"])}/{fmt(vitals["diastolic_bp"])} mmHg
- Mean arterial pressure: {fmt(vitals["mean_arterial_pressure"])}
- BMI: {fmt(vitals["bmi"])}

Comorbidities:
- CCI score: {fmt(comorb["cci_score"])}
- Recorded comorbidities: {comorb_text}

Respiratory/neurologic status:
- Type II respiratory failure: {fmt(resp["type_ii_respiratory_failure"])}
- Consciousness: {fmt(resp["consciousness"])}
- GCS: {fmt(resp["gcs"])}
- Oxygen inhalation: {fmt(resp["oxygen_inhalation"])}
- FiO2: {fmt(resp["fio2"])}
- Acute renal failure: {fmt(resp["acute_renal_failure"])}

Key labs:
- Creatinine: {fmt(labs["renal"]["creatinine_enzymatic_method"])}
- Urea: {fmt(labs["renal"]["urea"])}
- eGFR: {fmt(labs["renal"]["glomerular_filtration_rate"])}
- Sodium: {fmt(labs["electrolytes"]["sodium"])}
- Potassium: {fmt(labs["electrolytes"]["potassium"])}
- Chloride: {fmt(labs["electrolytes"]["chloride"])}
- Hemoglobin: {fmt(labs["cbc"]["hemoglobin"])}
- White blood cell count: {fmt(labs["cbc"]["white_blood_cell"])}
- Platelet count: {fmt(labs["cbc"]["platelet"])}
- BNP: {fmt(labs["cardiac_biomarkers"]["brain_natriuretic_peptide"])}
- High-sensitivity troponin: {fmt(labs["cardiac_biomarkers"]["high_sensitivity_troponin"])}
- D-dimer: {fmt(labs["coagulation"]["d_dimer"])}
- Albumin: {fmt(labs["nutrition_liver"]["albumin"])}

Medication summary:
- Medication count: {fmt(meds["med_count"])}
- Medications: {med_text}
""".strip()

    return text


patient_state_text_rows = []

for state in patient_states:
    patient_state_text_rows.append({
        "inpatient.number": state["patient_id"],
        "patient_state_text": patient_state_to_text(state)
    })

patient_state_text = pd.DataFrame(patient_state_text_rows)

patient_state_text_path = f"{STATE_DIR}/patient_state_text.csv"
patient_state_text.to_csv(patient_state_text_path, index=False)

display(patient_state_text.head())
print("Saved:", patient_state_text_path)

,inpatient.number,patient_state_text
0,857781,Patient State Summary:\n- Patient ID: 857781\n...
1,743087,Patient State Summary:\n- Patient ID: 743087\n...
2,866418,Patient State Summary:\n- Patient ID: 866418\n...
3,775928,Patient State Summary:\n- Patient ID: 775928\n...
4,810128,Patient State Summary:\n- Patient ID: 810128\n...


Saved: /content/drive/MyDrive/llm/patient_states/patient_state_text.csv


In [ ]:
def patient_state_to_compact_text(state):
    ctx = state["patient_context"]

    d = ctx["demographics"]
    e = ctx["encounter_context"]
    hf = ctx["heart_failure_status"]
    v = ctx["vitals_and_body"]
    c = ctx["comorbidities"]
    labs = ctx["key_labs"]
    meds = ctx["medications"]

    comorb_list = c.get("present_comorbidities", [])
    comorb_text = ", ".join(comorb_list) if len(comorb_list) > 0 else "none recorded"

    med_names = meds.get("medication_names", [])
    med_text = ", ".join(med_names[:8]) if len(med_names) > 0 else "not available"

    text = (
        f"Patient {fmt(state['patient_id'])}: "
        f"{fmt(d['gender'])}, age category {fmt(d['age_category'])}. "
        f"HF type {fmt(hf['type_of_heart_failure'])}, NYHA {fmt(hf['nyha_class'])}, Killip {fmt(hf['killip_grade'])}, "
        f"LVEF {fmt(hf['lvef'])}. "
        f"BP {fmt(v['systolic_bp'])}/{fmt(v['diastolic_bp'])}, pulse {fmt(v['pulse_bpm'])}, RR {fmt(v['respiration_rate'])}, "
        f"BMI {fmt(v['bmi'])}. "
        f"Comorbidities: {comorb_text}. "
        f"Creatinine {fmt(labs['renal']['creatinine_enzymatic_method'])}, eGFR {fmt(labs['renal']['glomerular_filtration_rate'])}, "
        f"Na {fmt(labs['electrolytes']['sodium'])}, K {fmt(labs['electrolytes']['potassium'])}, "
        f"Hb {fmt(labs['cbc']['hemoglobin'])}, BNP {fmt(labs['cardiac_biomarkers']['brain_natriuretic_peptide'])}, "
        f"troponin {fmt(labs['cardiac_biomarkers']['high_sensitivity_troponin'])}. "
        f"Meds: {med_text}."
    )

    return text


compact_rows = []

for state in patient_states:
    compact_rows.append({
        "inpatient.number": state["patient_id"],
        "patient_state_compact": patient_state_to_compact_text(state)
    })

patient_state_compact = pd.DataFrame(compact_rows)

patient_state_compact_path = f"{STATE_DIR}/patient_state_compact.csv"
patient_state_compact.to_csv(patient_state_compact_path, index=False)

display(patient_state_compact.head())
print("Saved:", patient_state_compact_path)

,inpatient.number,patient_state_compact
0,857781,"Patient 857781: Male, age category (69,79]. HF..."
1,743087,"Patient 743087: Female, age category (69,79]. ..."
2,866418,"Patient 866418: Male, age category (59,69]. HF..."
3,775928,"Patient 775928: Male, age category (69,79]. HF..."
4,810128,"Patient 810128: Female, age category (69,79]. ..."


Saved: /content/drive/MyDrive/llm/patient_states/patient_state_compact.csv


In [ ]:
print("Number of JSON patient states:", len(patient_states))
print("Patient state text rows:", patient_state_text.shape)
print("Compact patient state rows:", patient_state_compact.shape)

print("\nExample full patient-state text:\n")
print(patient_state_text.iloc[0]["patient_state_text"])

print("\nExample compact patient-state text:\n")
print(patient_state_compact.iloc[0]["patient_state_compact"])